In [1]:
pip install pandas numpy scikit-learn matplotlib seaborn openpyxl shap xgboost joblib


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import json
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from sklearn.metrics import roc_curve
from xgboost import XGBClassifier

In [3]:
import json
import pandas as pd

def load_yelp_data(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

def load_external_data(census_path, bls_path):
    census = pd.read_csv(census_path)
    bls = pd.read_excel(bls_path)
    return census, bls


yelp = load_yelp_data(
    r"C:\Users\pranu\Desktop\UMBC\fourth sem\business_feasibility\data\yelp_business.json"
)

In [16]:
yelp = load_yelp_data(
    r"C:\Users\pranu\Desktop\UMBC\fourth sem\business_feasibility\data\yelp_business.json"
)

census, bls = load_external_data(
    r"C:\Users\pranu\Desktop\UMBC\fourth sem\business_feasibility\data\census_city.csv",
    r"C:\Users\pranu\Desktop\UMBC\fourth sem\business_feasibility\data\bls_city.xlsx"
)

In [17]:
yelp = clean_yelp(yelp)

In [18]:
city_metrics = aggregate_city_metrics(yelp)

In [27]:
import re

def norm_city(x: str) -> str:
    if x is None:
        return ""
    s = str(x).strip().upper()

    # drop " city" suffix if it exists
    s = re.sub(r"\s+CITY$", "", s)

    # remove anything after comma (e.g., ", AZ")
    s = s.split(",")[0].strip()

    # if "AB EDMONTON" format, keep only last token(s) after 2-letter prefix
    s = re.sub(r"^[A-Z]{2}\s+", "", s)

    # collapse multiple spaces
    s = re.sub(r"\s+", " ", s)

    return s

In [28]:
import re
import pandas as pd

def clean_census_city_income(census: pd.DataFrame) -> pd.DataFrame:
    
    # pick only columns that are "Median income (dollars)!!Estimate"
    income_cols = [c for c in census.columns if "Median income (dollars)!!Estimate" in str(c)]
    if not income_cols:
        raise ValueError("No 'Median income (dollars)!!Estimate' columns found in census.")

    income_block = census[income_cols]
    row_idx = income_block.notna().sum(axis=1).idxmax()

    # build long format
    rows = []
    for col in income_cols:
        raw_city = col.split("!!")[0]  # e.g. "Phoenix city, Arizona"
        val = census.loc[row_idx, col]

        # clean value like "110,074" or "83,500"
        if pd.isna(val):
            continue
        val = str(val).replace(",", "").strip()
        # keep digits only
        m = re.search(r"(\d+)", val)
        if not m:
            continue

        rows.append({"city": raw_city, "median_income": float(m.group(1))})

    out = pd.DataFrame(rows)

    out["city_full"] = out["city"]
    out["city"] = out["city"].str.split(",").str[0].str.replace(r"\s+city$", "", regex=True).str.strip().str.upper()

    return out[["city", "median_income", "city_full"]]

In [34]:
import pandas as pd
import numpy as np
import re

def clean_bls_unemployment(bls: pd.DataFrame) -> pd.DataFrame:
    df = bls.copy().dropna(how="all")

    def row_has_2024(row) -> bool:
        for v in row:
            if pd.isna(v):
                continue
            if isinstance(v, (int, float, np.integer, np.floating)) and int(v) == 2024:
                return True
            s = str(v).strip()
            if s in {"2024", "2024.0"}:
                return True
        return False

    year_row = None
    for i in df.index:
        if row_has_2024(df.loc[i].values):
            year_row = i
            break
    if year_row is None:
        raise ValueError("Could not find a header row containing year 2024 in BLS sheet.")

    # set header
    header = [str(x).strip() if not pd.isna(x) else "" for x in df.loc[year_row].tolist()]
    df2 = df.loc[year_row + 1:].copy()
    df2.columns = header
    df2 = df2.loc[:, [c for c in df2.columns if c != ""]]

    # find 2024 col
    year_col = None
    for c in df2.columns:
        if str(c).strip() in {"2024", "2024.0"}:
            year_col = c
            break
    if year_col is None:
        for c in df2.columns:
            try:
                if int(float(str(c))) == 2024:
                    year_col = c
                    break
            except:
                pass
    if year_col is None:
        raise ValueError(f"No 2024 column found. Columns: {list(df2.columns)}")
    
    best_col = None
    best_score = -1

    for c in df2.columns:
        if c == year_col:
            continue
        sample = df2[c].dropna().astype(str).head(200)

        if sample.empty:
            continue

        # score signals
        has_state_pattern = sample.str.contains(r",\s*[A-Z]{2}\b").mean()          # "..., AZ"
        has_hyphen_metro = sample.str.contains(r"-").mean()                        # "Phoenix-Mesa-..."
        mostly_digits = sample.str.fullmatch(r"\d+").mean()                        # '01' '04' ...

        score = (2.0 * has_state_pattern) + (1.0 * has_hyphen_metro) - (3.0 * mostly_digits)

        if score > best_score:
            best_score = score
            best_col = c

    if best_col is None or best_score < 0:
        raise ValueError(
            "Could not reliably detect the metro/area name column in BLS sheet. "
            f"Columns seen: {list(df2.columns)}"
        )

    out = df2[[best_col, year_col]].copy()
    out = out.rename(columns={best_col: "area", year_col: "unemployment_rate_2024"})

    out["unemployment_rate_2024"] = pd.to_numeric(out["unemployment_rate_2024"], errors="coerce")
    out = out.dropna(subset=["unemployment_rate_2024"])

    out["employment_rate"] = 100.0 - out["unemployment_rate_2024"]

    # Convert area -> city key (first chunk)
    out["city"] = (
        out["area"].astype(str)
        .str.split(",").str[0]
        .str.split("-").str[0]
        .str.strip()
        .str.upper()
    )

    return out[["city", "employment_rate", "unemployment_rate_2024", "area"]]

In [35]:
bls_clean = clean_bls_unemployment(bls)
print("BLS cleaned cities:", bls_clean["city"].head(10).tolist())
print("Overlap count:", len(set(census_clean["city"]).intersection(set(bls_clean["city"]))))

BLS cleaned cities: ['BIRMINGHAM', 'PHOENIX', 'TUCSON', 'FRESNO', 'LOS ANGELES', 'RIVERSIDE', 'SACRAMENTO', 'SAN DIEGO', 'SAN FRANCISCO', 'SAN JOSE']
Overlap count: 9


In [41]:
def clean_yelp(df):
    df = df[df['is_open'] == 1]
    df = df[['city', 'stars', 'review_count']]
    df = df.dropna()
    return df

def aggregate_city_metrics(yelp_df):
    city_metrics = yelp_df.groupby('city').agg({
        'stars': ['mean', 'count'],
        'review_count': 'mean'
    }).reset_index()

    city_metrics.columns = ['city', 'avg_rating', 'business_count', 'avg_review_count']
    return city_metrics

yelp["city"] = yelp["city"].astype(str).str.strip().str.upper()

In [42]:
census_clean = clean_census_city_income(census)
bls_clean = clean_bls_unemployment(bls)

print("Census cleaned cities:", census_clean["city"].head(10).tolist())
print("BLS cleaned cities:", bls_clean["city"].head(10).tolist())

print("Census count:", len(census_clean), "BLS count:", len(bls_clean))
print("Overlap count:", len(set(census_clean["city"].apply(norm_city)).intersection(set(bls_clean["city"].apply(norm_city)))))

Census cleaned cities: ['CHANDLER', 'MESA', 'PHOENIX', 'ANAHEIM', 'LONG BEACH', 'LOS ANGELES', 'WILMINGTON', 'FORT LAUDERDALE', 'MIAMI', 'WEST PALM BEACH']
BLS cleaned cities: ['BIRMINGHAM', 'PHOENIX', 'TUCSON', 'FRESNO', 'LOS ANGELES', 'RIVERSIDE', 'SACRAMENTO', 'SAN DIEGO', 'SAN FRANCISCO', 'SAN JOSE']
Census count: 26 BLS count: 56
Overlap count: 9


In [43]:
def merge_datasets(city_metrics, census, bls):
    census_clean = clean_census_city_income(census)
    bls_clean = clean_bls_unemployment(bls)

    cm = city_metrics.copy()
    cm["city"] = cm["city"].astype(str).str.strip().str.upper()

    overlap = set(census_clean["city"]).intersection(set(bls_clean["city"]))
    if not overlap:
        raise ValueError("No overlap between Census and BLS after cleaning.")

    # keep only cities that exist in BOTH Census and BLS
    cm = cm[cm["city"].isin(overlap)]

    df = cm.merge(census_clean[["city", "median_income"]], on="city", how="inner")
    df = df.merge(bls_clean[["city", "employment_rate"]], on="city", how="inner")

    if df.empty:
        raise ValueError(
            "Merge produced 0 rows after filtering. "
        )

    print("✅ Merge rows:", len(df), "Cities:", sorted(df["city"].unique()))
    return df

In [44]:
def merge_datasets(city_metrics, census, bls):
    census_clean = clean_census_city_income(census)
    bls_clean = clean_bls_unemployment(bls)

    # normalize city keys everywhere
    city_metrics = city_metrics.copy()
    city_metrics["city_key"] = city_metrics["city"].apply(norm_city)

    census_clean = census_clean.copy()
    census_clean["city_key"] = census_clean["city"].apply(norm_city)

    bls_clean = bls_clean.copy()
    bls_clean["city_key"] = bls_clean["city"].apply(norm_city)

    # compute overlap set
    overlap = set(census_clean["city_key"]).intersection(set(bls_clean["city_key"]))
    if not overlap:
        raise ValueError("No overlap between Census and BLS city keys. Your BLS/Census parsing needs adjustment.")

    # filter Yelp metrics to only overlapping cities (THIS is the key fix)
    city_metrics = city_metrics[city_metrics["city_key"].isin(overlap)]

    # now merge
    df = city_metrics.merge(
        census_clean[["city_key", "median_income"]],
        on="city_key",
        how="inner"
    ).merge(
        bls_clean[["city_key", "employment_rate"]],
        on="city_key",
        how="inner"
    )

    if df.empty:
        # print diagnostics so you can see why
        cm_set = set(city_metrics["city_key"].head(50))
        cen_set = set(census_clean["city_key"].head(50))
        bls_set = set(bls_clean["city_key"].head(50))
        raise ValueError(
            "Merge still produced 0 rows.\n"
            f"Sample Yelp city_key: {list(cm_set)[:10]}\n"
            f"Sample Census city_key: {list(cen_set)[:10]}\n"
            f"Sample BLS city_key: {list(bls_set)[:10]}\n"
        )

    # keep a readable city column
    df["city"] = df["city_key"]

    return df

In [40]:
from sklearn.preprocessing import MinMaxScaler

def create_feasibility_score(df):

    # safety check
    if df.empty:
        raise ValueError("final_df is empty before feasibility scoring.")

    scaler = MinMaxScaler()

    features = [
        'avg_rating',
        'avg_review_count',
        'business_count',
        'median_income',
        'employment_rate'
    ]

    df[features] = scaler.fit_transform(df[features])

    df['feasibility_score'] = (
        0.25 * df['avg_rating'] +
        0.20 * df['avg_review_count'] +
        0.20 * df['business_count'] +
        0.20 * df['median_income'] +
        0.15 * df['employment_rate']
    )

    return df

In [7]:
def create_class_labels(df):

    q1 = df['feasibility_score'].quantile(0.33)
    q2 = df['feasibility_score'].quantile(0.66)

    def label(score):
        if score <= q1:
            return 0  # Low
        elif score <= q2:
            return 1  # Medium
        else:
            return 2  # High

    df['feasibility_class'] = df['feasibility_score'].apply(label)
    return df

In [47]:
from sklearn.model_selection import cross_val_score

def train_models(df):

    features = [
        'avg_rating',
        'avg_review_count',
        'business_count',
        'median_income',
        'employment_rate'
    ]

    X = df[features]
    y = df['class_label']

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC

    models = {
        "RandomForest": RandomForestClassifier(),
        "LogisticRegression": LogisticRegression(),
        "SVM": SVC()
    }

    best_model = None
    best_score = 0

    # 🔹 adjust CV folds automatically
    cv_folds = min(5, len(X_train))

    for name, model in models.items():

        if cv_folds > 1:
            scores = cross_val_score(model, X_train, y_train, cv=cv_folds)
            mean_score = scores.mean()
        else:
            mean_score = 0

        print(f"{name} CV Accuracy: {mean_score:.4f}")

        if mean_score > best_score:
            best_score = mean_score
            best_model = model

    best_model.fit(X_train, y_train)

    return best_model, X_test, y_test

In [9]:
def plot_feature_importance(model, feature_names):

    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_

        plt.figure(figsize=(8,5))
        sns.barplot(x=importance, y=feature_names)
        plt.title("Feature Importance")
        plt.show()

In [10]:
def shap_explain(model, X_sample):

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    shap.summary_plot(shap_values, X_sample)

In [11]:
def save_model(model):
    joblib.dump(model, "models/feasibility_model.pkl")
    print("Model Saved Successfully!")

In [45]:
print("city_metrics columns:", list(city_metrics.columns))
print("census columns:", list(census.columns))
print("bls columns:", list(bls.columns))

print("\nSample city_metrics:\n", city_metrics.head(3))
print("\nSample census:\n", census.head(3))
print("\nSample bls:\n", bls.head(3))

city_metrics columns: ['city', 'avg_rating', 'business_count', 'avg_review_count']
census columns: ['Label (Grouping)', 'Chandler city, Arizona!!Number!!Estimate', 'Chandler city, Arizona!!Number!!Margin of Error', 'Chandler city, Arizona!!Percent Distribution!!Estimate', 'Chandler city, Arizona!!Percent Distribution!!Margin of Error', 'Chandler city, Arizona!!Median income (dollars)!!Estimate', 'Chandler city, Arizona!!Median income (dollars)!!Margin of Error', 'Mesa city, Arizona!!Number!!Estimate', 'Mesa city, Arizona!!Number!!Margin of Error', 'Mesa city, Arizona!!Percent Distribution!!Estimate', 'Mesa city, Arizona!!Percent Distribution!!Margin of Error', 'Mesa city, Arizona!!Median income (dollars)!!Estimate', 'Mesa city, Arizona!!Median income (dollars)!!Margin of Error', 'Phoenix city, Arizona!!Number!!Estimate', 'Phoenix city, Arizona!!Number!!Margin of Error', 'Phoenix city, Arizona!!Percent Distribution!!Estimate', 'Phoenix city, Arizona!!Percent Distribution!!Margin of Erro

In [48]:
def main():

    print("Loading Data...")

    yelp = load_yelp_data(
        r"C:\Users\pranu\Desktop\UMBC\fourth sem\business_feasibility\data\yelp_business.json"
    )

    census, bls = load_external_data(
        r"C:\Users\pranu\Desktop\UMBC\fourth sem\business_feasibility\data\census_city.csv",
        r"C:\Users\pranu\Desktop\UMBC\fourth sem\business_feasibility\data\bls_city.xlsx"
    )

    yelp = clean_yelp(yelp)
    city_metrics = aggregate_city_metrics(yelp)
    final_df = merge_datasets(city_metrics, census, bls)

    final_df = create_feasibility_score(final_df)
    final_df = create_class_labels(final_df)

    model, X_test, y_test = train_models(final_df)

    print("\nClassification Report:")
    print(classification_report(y_test, model.predict(X_test)))

    plot_feature_importance(
        model,
        ['avg_rating','avg_review_count','business_count','median_income','employment_rate']
    )

    shap_explain(model, X_test)

    save_model(model)

    final_df.to_csv("final_results.csv", index=False)


if __name__ == "__main__":
    main()

Loading Data...


KeyError: 'class_label'